<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_66_distributed_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🛰️ **Module 66 — Distributed Training (DDP / FSDP / DeepSpeed)** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)

> Phase 9 · Production Gaps


# Module 66 — Distributed Training

> M57 trained a tiny GPT on a single CPU. M39 fine-tuned a 7B on one GPU with Unsloth. **Real** pretraining and serious fine-tuning need **many GPUs talking to each other**. This module is the map of how that's done: **DDP**, **FSDP**, **DeepSpeed ZeRO**, plus **tensor + pipeline parallelism** for frontier-scale models. You won't run a 4-GPU job from Colab — but every snippet here is production-shaped code you can drop onto a real cluster.

### What you'll cover
1. **Why** you need distribution — the memory math
2. **`torch.distributed` primitives** — process groups, all-reduce, NCCL
3. **DDP** — Distributed Data Parallel (the workhorse)
4. **FSDP** — Fully Sharded Data Parallel (the workhorse for ≥7B)
5. **DeepSpeed ZeRO** — stages 1 / 2 / 3 explained as one knob
6. **Tensor + Pipeline parallelism** — when one model > one GPU
7. **3D parallelism** — what frontier labs actually run
8. **Launchers** — `torchrun` · `accelerate` · `deepspeed`
9. **Gradient accumulation, gradient checkpointing, AMP** — the small wins that compose
10. Practical recipe — fine-tune a 7B Llama on 4× A100 with FSDP


## 1 · The memory math

To train a model in fp16/bf16 with AdamW you need, **per parameter**:

| Tensor | Bytes / param (mixed precision) | Why |
|---|---|---|
| Weights (fp16) | 2 | the model |
| Gradients (fp16) | 2 | one per backward pass |
| **AdamW state — momentum (fp32)** | 4 | first moment |
| **AdamW state — variance (fp32)** | 4 | second moment |
| **Master weights (fp32)** | 4 | for accurate update |
| **Activations** | varies | depends on sequence length & checkpointing |
| **= "16 bytes / param" rule of thumb** | **16** | + activations |

So a **7 B** model needs **~112 GB** just for weights+grads+optimiser — already more than a single 80 GB A100. **70 B** is **~1.1 TB**. There's no single GPU that fits it. You **must** distribute.

| Model | Just weights+grads+opt (fp16 + fp32 AdamW) | Fits on… |
|---|---|---|
| 1B  | ~16 GB | one consumer GPU |
| 7B  | ~112 GB | 2× A100 80GB (with sharding) |
| 13B | ~208 GB | 4× A100 |
| 70B | ~1.1 TB | 16× A100 / 8× H100 |
| 405B | ~6.5 TB | 96× H100 |

## 2 · `torch.distributed` primitives — the layer everything sits on

In [ ]:
ddp_setup_sketch = '''
import os, torch, torch.distributed as dist

# every process learns its identity from env vars set by the launcher
local_rank = int(os.environ["LOCAL_RANK"])      # 0..gpus_per_node-1
world_size = int(os.environ["WORLD_SIZE"])      # total GPUs across all nodes
rank       = int(os.environ["RANK"])            # global rank

# initialise the process group over NCCL (or "gloo" on CPU-only)
dist.init_process_group(backend="nccl")
torch.cuda.set_device(local_rank)
'''
print(ddp_setup_sketch)

**The five collective operations everything else is built on.**

| Operation | What it does | Where it shows up |
|---|---|---|
| **`all_reduce(t)`** | sum (or avg) `t` across every rank, write the result back to every rank | DDP gradient sync |
| **`broadcast(t, src)`** | rank `src` sends `t` to everyone | spreading initial weights |
| **`all_gather`** | every rank collects everyone's tensor into a list | FSDP unsharding |
| **`reduce_scatter`** | sum across ranks, then keep only a shard on each | FSDP gradient sharding |
| **`barrier`** | every rank waits for everyone else | sync points |

**NCCL** (NVIDIA Collective Communications Library) implements all of these over **NVLink** (within a node, ~600 GB/s) and **InfiniBand** (between nodes, ~50-400 GB/s). NCCL is fast specifically because it uses NVLink topology — that's why ML cluster wiring matters so much.

## 3 · DDP — Distributed Data Parallel

**The simplest distributed strategy.** Each GPU keeps a full copy of the model. Each batch is split across GPUs (one micro-batch per GPU). After backward, gradients are **all-reduced** across GPUs (averaged). Optimiser steps locally, in lockstep with everyone else.

```
   GPU 0:  full model + grads + opt state    ←┐
   GPU 1:  full model + grads + opt state    │ all-reduce gradients every step
   GPU 2:  full model + grads + opt state    │
   GPU 3:  full model + grads + opt state    ←┘
```

In [ ]:
ddp_loop = '''
import torch
import torch.nn as nn
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data.distributed import DistributedSampler

dist.init_process_group(backend="nccl")
local_rank = int(os.environ["LOCAL_RANK"])
torch.cuda.set_device(local_rank)

model = MyTransformer().cuda(local_rank)
model = DDP(model, device_ids=[local_rank])              # wraps the model

sampler = DistributedSampler(dataset, shuffle=True)      # each rank sees a unique slice
loader  = DataLoader(dataset, batch_size=32, sampler=sampler)

optim = torch.optim.AdamW(model.parameters(), lr=3e-5)

for epoch in range(num_epochs):
    sampler.set_epoch(epoch)                             # different shuffle each epoch
    for x, y in loader:
        x = x.cuda(local_rank); y = y.cuda(local_rank)
        optim.zero_grad()
        loss = model(x, labels=y).loss                    # forward through DDP-wrapped model
        loss.backward()                                   # backward → DDP all-reduces grads
        optim.step()
'''
print(ddp_loop)

**Three places DDP differs from a single-GPU loop.**
1. `DDP(model)` — wrap once. After this, `loss.backward()` triggers an asynchronous all-reduce of every gradient.
2. `DistributedSampler` — gives each rank a unique slice of the dataset (no overlap).
3. `sampler.set_epoch(epoch)` — required, otherwise every epoch sees the same shuffle.

**When DDP wins.** Model fits on one GPU; you want linear throughput scale-out. This is most fine-tuning workloads under ~7B.

**When DDP breaks.** Model *doesn't* fit on one GPU — every rank still holds the full weights+grads+optimiser. That's where FSDP comes in.

## 4 · FSDP — Fully Sharded Data Parallel

**The 2022+ default for large-model training.** Instead of replicating the model on every GPU, **shard** the parameters, gradients, and optimiser state across all GPUs. Each rank holds only its **slice** of each tensor; before a layer's forward pass it `all_gather`s the full weights from every other rank, runs the forward, then immediately re-shards.

```
   layer i weights:    [shard₀, shard₁, shard₂, shard₃]
                        GPU 0   GPU 1   GPU 2   GPU 3
   forward i:   all_gather → full weights on every rank → matmul → re-shard
   backward i:  full weights → grad → reduce_scatter → per-rank grad shard
```

Same final result as DDP; **`N×` lower memory footprint per GPU**. The cost is extra communication — every forward and backward layer triggers a gather/scatter.

In [ ]:
fsdp_setup = '''
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
from torch.distributed.fsdp import MixedPrecision, BackwardPrefetch
from torch.distributed.fsdp.wrap import transformer_auto_wrap_policy
from functools import partial

mp_policy = MixedPrecision(
    param_dtype  = torch.bfloat16,
    reduce_dtype = torch.bfloat16,
    buffer_dtype = torch.bfloat16,
)

# Wrap one transformer block at a time so each block can be re-sharded independently
auto_wrap = partial(
    transformer_auto_wrap_policy,
    transformer_layer_cls={ TransformerBlock },              # your block class from M56/M61
)

model = FSDP(
    model.cuda(),
    auto_wrap_policy=auto_wrap,
    mixed_precision=mp_policy,
    backward_prefetch=BackwardPrefetch.BACKWARD_PRE,         # overlap comms with compute
)
'''
print(fsdp_setup)

**FSDP knobs that matter.**
- **`auto_wrap_policy`** — at what granularity to shard. Per-transformer-block is the default for LLMs.
- **`MixedPrecision`** — store params in bf16, do the reduce in bf16. Saves bandwidth + memory.
- **`BackwardPrefetch`** — overlap the gather of layer `i-1` with the backward of layer `i`. Big throughput win.
- **`activation_checkpointing`** — recompute activations during backward instead of storing them. Trades 1× FLOPs for ~10× activation memory.
- **`CPUOffload(offload_params=True)`** — keep param shards on CPU when not in use. For when you're memory-starved even after sharding.

## 5 · DeepSpeed ZeRO — same idea, different knob

DeepSpeed's **ZeRO** (Zero Redundancy Optimizer) predated FSDP and is the *other* dominant strategy. It exposes the sharding granularity as **three stages**:

| Stage | Shards | Memory ratio | Notes |
|---|---|---|---|
| **ZeRO-1** | optimiser state only | ~4× saving | grads + params still replicated |
| **ZeRO-2** | + gradients | ~8× saving | params still replicated |
| **ZeRO-3** | + parameters | full DDP→FSDP equivalent | shard everything, just like FSDP |
| **ZeRO-Infinity** | + offload to NVMe | ~10× more | for truly massive models on small clusters |

**FSDP ≈ ZeRO-3.** They are mathematically the same trick implemented by different teams. Use whichever your codebase already speaks; both are first-class in HuggingFace `accelerate` and `transformers`.

```jsonc
// deepspeed_config.json
{
  "zero_optimization": {
    "stage": 3,
    "offload_optimizer": { "device": "cpu" },
    "overlap_comm": true
  },
  "bf16": { "enabled": true },
  "gradient_accumulation_steps": 8,
  "train_micro_batch_size_per_gpu": 2
}
```

DeepSpeed extras you won't find in vanilla FSDP:
- **DeepSpeed-Inference** — heavy quantisation + tensor-parallel kernels for inference.
- **DeepSpeed-Chat** — full RLHF pipeline (SFT + RM + PPO) with one config.
- **DeepSpeed-Ulysses** — sequence parallelism for very long context.

## 6 · Tensor + Pipeline parallelism — when one *layer* > one GPU

DDP/FSDP/ZeRO solve memory by sharding across the **batch** dimension (data-parallel). They all assume one **layer's forward pass** fits on one GPU. For frontier models (70 B in fp32, 405 B at any precision) that's not true. Two further axes:

### Tensor Parallelism (TP) — Megatron-LM
Split each matrix multiply across `k` GPUs. The Q projection in attention becomes `Q = x @ W_q`; with TP=2, we split `W_q` column-wise and run `Q_part = x @ W_q_shard` on each of the 2 GPUs, then `all_gather` the result.

```
   x  → matmul split column-wise  → all_gather → next layer
   GPU0: x @ W[:, 0:d/2]
   GPU1: x @ W[:, d/2:d]
```

Used inside one node where NVLink is fast.

### Pipeline Parallelism (PP) — GPipe / Megatron-PP
Split **layers** across `k` GPUs in a pipeline. GPU 0 runs layers 0-23, GPU 1 runs 24-47, etc. A batch of size `B` is broken into `B/m` micro-batches that flow through the pipeline like an assembly line — keeps every stage busy.

Used **between** nodes where the slower inter-node bandwidth can't sustain TP.

## 7 · 3D parallelism — what frontier labs run

For **Llama 3.1 405B** / **DeepSeek-V3** / **GPT-4-class** models you combine all three axes:

```
                ┌──── data-parallel replicas (FSDP / ZeRO-3) ────┐
                ▼                                                ▼
   ┌─ tensor-parallel within node (8× H100, NVLink) ──┐    ┌─ TP ─┐
   │       ┌── pipeline stages across nodes ──┐       │ ←─ │      │
   │       ▼                                  ▼       │    │      │
   │  layers 0-31  layers 32-63  layers 64-95 │       │    │      │
   └──────────────────────────────────────────┘       │    └──────┘
```

| Axis | Splits across | Shrinks |
|---|---|---|
| **Data** | batch | optimiser + grads + (optionally) params |
| **Tensor** | matrix columns / rows | params per layer |
| **Pipeline** | layers | params per stage |

For a 405 B model you might use **`TP=8`, `PP=16`, `DP=4`** → 512 GPUs. Each axis is tuned for the underlying network topology: TP within node (NVLink), PP between nodes (InfiniBand fat-tree), DP at the top.

**Frameworks that do this end-to-end:** **Megatron-LM**, **NeMo Megatron**, **veRL**, **TorchTitan**, **OpenRLHF**.

## 8 · Launchers — how the processes actually start

You never call `python my_train.py`. You launch with one of these:

### `torchrun` (PyTorch built-in)
```bash
torchrun \
    --nproc_per_node=4 \
    --nnodes=2 \
    --node_rank=0 \
    --master_addr=10.0.0.1 --master_port=29500 \
    train.py --batch_size 32
```

Spawns one process per GPU on each node, sets `LOCAL_RANK`/`RANK`/`WORLD_SIZE`.

### `accelerate` (HuggingFace)
```bash
accelerate config           # one-time interactive config
accelerate launch train.py   # uses the saved config
```

Same idea, plus a high-level Python API (`Accelerator()`) that wraps your model/optimiser and **chooses DDP / FSDP / DeepSpeed for you** based on the config. **The path of least resistance for most teams.**

### `deepspeed`
```bash
deepspeed --num_gpus=4 train.py --deepspeed_config ds_config.json
```

Launcher + DeepSpeed engine in one binary.

### SLURM / Kubernetes
Production clusters use SLURM (HPC) or Kubernetes operators (Kubeflow Training Operator from M46) to allocate nodes and call `torchrun` automatically. The `torchrun` command is what matters — the rest is scheduling glue.

## 9 · The small wins that compose

These are **orthogonal** to the parallelism strategy and stack on top of any of DDP/FSDP/DeepSpeed:

| Trick | What it saves | When to enable |
|---|---|---|
| **Mixed precision** (`bf16`/`fp16`) | 2× memory + 2-3× speed | always for modern GPUs |
| **Gradient accumulation** (`k=8`) | scales effective batch size on small GPUs | always — first knob to turn |
| **Gradient checkpointing** | ~10× activation memory at 1.33× compute | when you OOM on activations |
| **Flash Attention 2/3** | ~3× attention speed + linear memory | always (PyTorch 2.5+ wires it in for you) |
| **Fused optimisers** (`apex` or `torch._foreach_*`) | ~15% step speed | usually free; on by default with `accelerate` |
| **`torch.compile`** | ~10-30% speed for training | try once your code is stable |
| **Selective layer freezing** (M58) | huge for fine-tuning | when you're really doing LoRA / partial-FT |

Combining these you can fine-tune a **7B in bf16 + FSDP + grad-checkpointing + flash-attn** on **2× A100 40 GB** with batch size 8 × accumulation 16. Without them you'd need 8× the GPUs.

## 10 · Practical recipe — fine-tune Llama-3-8B on 4× A100

The realistic 2026 default for a 7-8B SFT or DPO run, using HuggingFace `accelerate` + `trl`:

```bash
# 1. Create accelerate config
$ accelerate config
# answer: multi-GPU, 4 GPUs, FSDP yes, transformer_auto_wrap, bf16

# 2. Or write the config by hand:
$ cat fsdp_config.yaml
distributed_type: FSDP
fsdp_config:
  fsdp_sharding_strategy: FULL_SHARD
  fsdp_auto_wrap_policy:  TRANSFORMER_BASED_WRAP
  fsdp_transformer_layer_cls_to_wrap: LlamaDecoderLayer
  fsdp_state_dict_type: SHARDED_STATE_DICT
  fsdp_offload_params: false
  fsdp_use_orig_params: true
mixed_precision: bf16
num_processes: 4

# 3. Launch
$ accelerate launch --config_file fsdp_config.yaml train.py \
    --model_name meta-llama/Meta-Llama-3-8B \
    --batch_size 4 --gradient_accumulation_steps 8 \
    --learning_rate 2e-5 --max_seq_len 4096
```

Inside `train.py` you just write a normal HuggingFace `Trainer` / `SFTTrainer` / `DPOTrainer` loop — `accelerate` injects FSDP underneath.

**Knobs you'll tune in this order.**
1. **Batch size per GPU** — start at whatever fits; bump until OOM.
2. **Gradient accumulation** — multiply effective batch size to your target.
3. **`max_seq_len`** — limits activation memory more than batch size for long contexts.
4. **Activation checkpointing** — when you OOM on activations.
5. **CPU offload** — last resort; very slow.

> 🟡 Final warning: **multi-GPU runs that "almost work" are the most expensive bug class in ML.** Always sanity-check on **1 GPU** first; then 2; then `N`. NaNs that only appear at 8 GPUs are usually grad reductions diverging — try `fp32` reductions before chasing data bugs.

## ✅ Recap
- "16 bytes/param" rule → 7B needs ~112 GB; 70B ~1.1 TB. You must distribute.
- **DDP** = replicate model on every GPU, all-reduce gradients. Use when the model fits.
- **FSDP** ≈ **ZeRO-3** = shard params + grads + opt state; gather-on-demand. Use for ≥7B.
- **TP + PP** add when one *layer* doesn't fit one GPU (frontier-scale).
- **3D parallelism** combines all three; tune each axis to the network topology.
- **Launchers**: `torchrun` → `accelerate` → `deepspeed`. `accelerate` chooses for you.
- Stack **bf16 + grad-accum + grad-checkpointing + flash-attn + `torch.compile`** on top of any strategy.
- Sanity-check on 1 GPU before scaling.

Next: **M67 — RLHF / GRPO**.
